In [ ]:
import tkinter as tk
from tkinter import messagebox, simpledialog, ttk, filedialog
from PIL import Image, ImageTk, ImageOps
import sqlite3
import hashlib
import os
import shutil
import datetime

DB_FILE = "nft_users.db"
STARTING_BALANCE = 1000.0
NFT_IMAGE_DIR = "nft_images"

if not os.path.exists(NFT_IMAGE_DIR):
    os.makedirs(NFT_IMAGE_DIR)

class DatabaseManager:
    def __init__(self, db_file):
        self.db_file = db_file
        self.conn = None
        self.connect()
        self.setup_database()

    def connect(self):
        try:
            self.conn = sqlite3.connect(self.db_file, timeout=10)
            self.conn.execute("PRAGMA journal_mode=WAL;")
            self.conn.row_factory = sqlite3.Row
        except sqlite3.Error as e:
            print(f"Error connecting to database: {e}")

    def setup_database(self):
        if self.conn is None:
            print("No database connection available for setup.")
            return
        try:
            cursor = self.conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS users (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    email TEXT UNIQUE NOT NULL,
                    password TEXT NOT NULL
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS nfts (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_id INTEGER NOT NULL,
                    name TEXT NOT NULL,
                    description TEXT,
                    price REAL NOT NULL,
                    image_path TEXT,
                    FOREIGN KEY (user_id) REFERENCES users(id)
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS wallets (
                    user_id INTEGER PRIMARY KEY,
                    balance REAL NOT NULL,
                    total_expenses REAL NOT NULL,
                    total_profit_loss REAL NOT NULL,
                    FOREIGN KEY (user_id) REFERENCES users(id)
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS transactions (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_id INTEGER NOT NULL,
                    nft_id INTEGER,
                    type TEXT NOT NULL,
                    amount REAL NOT NULL,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    FOREIGN KEY (user_id) REFERENCES users(id),
                    FOREIGN KEY (nft_id) REFERENCES nfts(id)
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS auctions (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    nft_id INTEGER UNIQUE NOT NULL,
                    seller_id INTEGER NOT NULL,
                    min_bid_price REAL NOT NULL,
                    current_bid REAL,
                    current_bidder INTEGER,
                    start_time DATETIME DEFAULT CURRENT_TIMESTAMP,
                    end_time DATETIME,
                    FOREIGN KEY (nft_id) REFERENCES nfts(id),
                    FOREIGN KEY (seller_id) REFERENCES users(id),
                    FOREIGN KEY (current_bidder) REFERENCES users(id)
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS bids (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    auction_id INTEGER NOT NULL,
                    bidder_id INTEGER NOT NULL,
                    bid_amount REAL NOT NULL,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    FOREIGN KEY (auction_id) REFERENCES auctions(id),
                    FOREIGN KEY (bidder_id) REFERENCES users(id)
                )
            """)
            self.conn.commit()
            cursor.close()
        except sqlite3.Error as e:
            print(f"Error creating tables: {e}")

    def add_user(self, email, password):
        if self.conn is None:
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute("INSERT INTO users (email, password) VALUES (?, ?)", (email, password))
            user_id = cursor.lastrowid
            cursor.execute(
                "INSERT INTO wallets (user_id, balance, total_expenses, total_profit_loss) VALUES (?, ?, 0, 0)",
                (user_id, STARTING_BALANCE)
            )
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.IntegrityError:
            return False
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def verify_user(self, email, password):
        if self.conn is None:
            return False, None
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT id, password FROM users WHERE email = ?", (email,))
            row = cursor.fetchone()
            cursor.close()
            if row:
                stored_password = row["password"]
                return stored_password == self.hash_password(password), row["id"]
            else:
                return False, None
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False, None

    def add_nft(self, user_id, name, description, price, image_path):
        if self.conn is None:
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute("INSERT INTO nfts (user_id, name, description, price, image_path) VALUES (?, ?, ?, ?, ?)",
                           (user_id, name, description, price, image_path))
            nft_id = cursor.lastrowid
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def get_nfts(self, filter_text=None):
        if self.conn is None:
            return []
        try:
            cursor = self.conn.cursor()
            if filter_text:
                filter_term = f"%{filter_text.lower()}%"
                cursor.execute(
                    "SELECT id, user_id, name, description, price, image_path FROM nfts WHERE LOWER(name) LIKE ? OR LOWER(description) LIKE ?",
                    (filter_term, filter_term)
                )
            else:
                cursor.execute("SELECT id, user_id, name, description, price, image_path FROM nfts")
            nfts = cursor.fetchall()
            cursor.close()
            return nfts
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return []

    def get_user_nfts(self, user_id):
        if self.conn is None:
            return []
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT id, name, description, price, image_path FROM nfts WHERE user_id = ?", (user_id,))
            nfts = cursor.fetchall()
            cursor.close()
            return nfts
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return []

    def remove_nft(self, nft_id):
        if self.conn is None:
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT image_path FROM nfts WHERE id = ?", (nft_id,))
            row = cursor.fetchone()
            if row and row["image_path"]:
                img_path = row["image_path"]
                if os.path.exists(img_path):
                    try:
                        os.remove(img_path)
                    except Exception:
                        pass
            cursor.execute("DELETE FROM auctions WHERE nft_id = ?", (nft_id,))
            cursor.execute("DELETE FROM nfts WHERE id = ?", (nft_id,))
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def get_wallet(self, user_id):
        if self.conn is None:
            return None
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (user_id,))
            row = cursor.fetchone()
            if not row:
                # create wallet record with starting balance if missing
                cursor.execute("INSERT INTO wallets (user_id, balance, total_expenses, total_profit_loss) VALUES (?, ?, 0, 0)", (user_id, STARTING_BALANCE))
                self.conn.commit()
                cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (user_id,))
                row = cursor.fetchone()
            cursor.close()
            if row:
                return {"balance": row["balance"], "expenses": row["total_expenses"], "profit_loss": row["total_profit_loss"]}
            else:
                return None
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return None

    def update_wallet(self, user_id, balance=None, expenses=None, profit_loss=None):
        if self.conn is None:
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (user_id,))
            row = cursor.fetchone()
            if not row:
                cursor.close()
                return False
            current_balance, current_expenses, current_profit_loss = row["balance"], row["total_expenses"], row["total_profit_loss"]
            new_balance = balance if balance is not None else current_balance
            new_expenses = expenses if expenses is not None else current_expenses
            new_profit_loss = profit_loss if profit_loss is not None else current_profit_loss
            cursor.execute("""
                UPDATE wallets SET balance=?, total_expenses=?, total_profit_loss=? WHERE user_id=?
            """, (new_balance, new_expenses, new_profit_loss, user_id))
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def add_transaction(self, user_id, nft_id, type_, amount):
        if self.conn is None:
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute("INSERT INTO transactions (user_id, nft_id, type, amount, timestamp) VALUES (?, ?, ?, ?, ?)",
                           (user_id, nft_id, type_, amount, datetime.datetime.now()))
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def get_transactions(self, user_id):
        if self.conn is None:
            return []
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT type, amount, timestamp, nft_id FROM transactions WHERE user_id = ? ORDER BY timestamp DESC", (user_id,))
            rows = cursor.fetchall()
            cursor.close()
            return rows
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return []

    def get_auction_by_nft(self, nft_id):
        try:
            cursor = self.conn.cursor()
            cursor.execute("""
                SELECT * FROM auctions WHERE nft_id = ?
            """, (nft_id,))
            auction = cursor.fetchone()
            cursor.close()
            return auction
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return None

    def add_auction(self, nft_id, seller_id, min_bid_price):
        try:
            cursor = self.conn.cursor()
            cursor.execute("""
                INSERT INTO auctions (nft_id, seller_id, min_bid_price, current_bid, current_bidder)
                VALUES (?, ?, ?, NULL, NULL)
            """, (nft_id, seller_id, min_bid_price))
            self.conn.commit()
            cursor.close()
            return True
        except sqlite3.IntegrityError:
            # Auction already exists for this NFT
            return False
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False

    def get_active_auctions(self):
        try:
            cursor = self.conn.cursor()
            cursor.execute("""
                SELECT a.id as auction_id, a.nft_id, a.seller_id, a.min_bid_price, a.current_bid, a.current_bidder,
                       n.name, n.description, n.price, n.image_path, u.email as seller_email
                FROM auctions a
                JOIN nfts n ON a.nft_id = n.id
                JOIN users u ON a.seller_id = u.id
            """)
            auctions = cursor.fetchall()
            cursor.close()
            return auctions
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return []

    def add_bid(self, auction_id, bidder_id, bid_amount):
        try:
            cursor = self.conn.cursor()
            # Check current highest bid
            cursor.execute("SELECT current_bid, min_bid_price FROM auctions WHERE id = ?", (auction_id,))
            row = cursor.fetchone()
            if not row:
                cursor.close()
                return False, "Auction not found."
            current_bid = row["current_bid"]
            min_bid = row["min_bid_price"]
            # Bid must be higher than current bid or at least min_bid if no bids
            if current_bid is None:
                if bid_amount < min_bid:
                    cursor.close()
                    return False, f"Bid must be at least the minimum bid price (${min_bid:.2f})."
            else:
                if bid_amount <= current_bid:
                    cursor.close()
                    return False, f"Bid must be higher than the current highest bid (${current_bid:.2f})."
            # Add bid into bids table
            cursor.execute("INSERT INTO bids (auction_id, bidder_id, bid_amount) VALUES (?, ?, ?)", (auction_id, bidder_id, bid_amount))
            # Update auctions table with new highest bid and bidder
            cursor.execute("UPDATE auctions SET current_bid = ?, current_bidder = ? WHERE id = ?", (bid_amount, bidder_id, auction_id))
            self.conn.commit()
            cursor.close()
            return True, "Bid placed successfully."
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False, "Database error"

    def get_bids_for_auction(self, auction_id):
        try:
            cursor = self.conn.cursor()
            cursor.execute("""
                SELECT b.bid_amount, b.timestamp, u.email as bidder_email
                FROM bids b
                JOIN users u ON b.bidder_id = u.id
                WHERE b.auction_id = ?
                ORDER BY b.bid_amount DESC
            """, (auction_id,))
            bids = cursor.fetchall()
            cursor.close()
            return bids
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return []

    def end_auction(self, auction_id):
        """
        Ends the auction, transfers NFT ownership to highest bidder if any,
        updates wallets and transactions. Removes auction record.
        """
        try:
            cursor = self.conn.cursor()
            cursor.execute("SELECT nft_id, seller_id, current_bid, current_bidder FROM auctions WHERE id = ?", (auction_id,))
            auction = cursor.fetchone()
            if not auction:
                cursor.close()
                return False, "Auction not found."
            current_bid = auction["current_bid"]
            current_bidder = auction["current_bidder"]
            nft_id = auction["nft_id"]
            seller_id = auction["seller_id"]

            # If no bids, just remove auction, NFT stays with seller
            if current_bid is None or current_bidder is None:
                cursor.execute("DELETE FROM auctions WHERE id = ?", (auction_id,))
                self.conn.commit()
                cursor.close()
                return True, "Auction ended with no bids. NFT remains with you."

            # Transfer NFT ownership to highest bidder
            cursor.execute("SELECT name, description, price, image_path FROM nfts WHERE id = ?", (nft_id,))
            nft_details = cursor.fetchone()

            if not nft_details:
                cursor.close()
                return False, "NFT details not found."

            # Remove NFT from seller
            cursor.execute("DELETE FROM nfts WHERE id = ?", (nft_id,))

            # Add NFT to highest bidder with price = current_bid
            cursor.execute("INSERT INTO nfts (user_id, name, description, price, image_path) VALUES (?, ?, ?, ?, ?)",
                           (current_bidder, nft_details["name"], nft_details["description"], current_bid, nft_details["image_path"]))
            new_nft_id = cursor.lastrowid

            # Update seller wallet: add sale amount to profit_loss and balance
            cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (seller_id,))
            wallet = cursor.fetchone()
            if wallet:
                new_balance = wallet["balance"] + current_bid
                new_profit_loss = wallet["total_profit_loss"] + current_bid
                cursor.execute("UPDATE wallets SET balance = ?, total_profit_loss = ? WHERE user_id = ?", (new_balance, new_profit_loss, seller_id))

            # Update buyer wallet: subtract current_bid from balance and add expenses
            cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (current_bidder,))
            wallet_buyer = cursor.fetchone()
            if wallet_buyer:
                new_balance_buyer = wallet_buyer["balance"] - current_bid
                new_expenses_buyer = wallet_buyer["total_expenses"] + current_bid
                cursor.execute("UPDATE wallets SET balance = ?, total_expenses = ? WHERE user_id = ?", (new_balance_buyer, new_expenses_buyer, current_bidder))

            # Add transactions for both seller and buyer
            cursor.execute("INSERT INTO transactions (user_id, nft_id, type, amount, timestamp) VALUES (?, ?, ?, ?, ?)",
                           (seller_id, nft_id, "sell", current_bid, datetime.datetime.now()))
            cursor.execute("INSERT INTO transactions (user_id, nft_id, type, amount, timestamp) VALUES (?, ?, ?, ?, ?)",
                           (current_bidder, new_nft_id, "buy", current_bid, datetime.datetime.now()))

            # Remove auction record and bids
            cursor.execute("DELETE FROM bids WHERE auction_id = ?", (auction_id,))
            cursor.execute("DELETE FROM auctions WHERE id = ?", (auction_id,))

            self.conn.commit()
            cursor.close()
            return True, "Auction ended successfully. NFT sold."
        except sqlite3.Error as e:
            messagebox.showerror("DB Error", f"Database error: {e}")
            return False, "Database error."

    @staticmethod
    def hash_password(password):
        return hashlib.sha256(password.encode('utf-8')).hexdigest()

    def close(self):
        if self.conn:
            self.conn.close()

class NFTCard(tk.Frame):
    def __init__(self, parent, nft_data, buy_callback=None, sell_callback=None, remove_callback=None, auction_callback=None, place_bid_callback=None, end_auction_callback=None, is_auction=False, *args, **kwargs):
        super().__init__(parent, bd=2, relief="groove", padx=10, pady=10, *args, **kwargs)
        self.nft_data = nft_data
        self.buy_callback = buy_callback
        self.sell_callback = sell_callback
        self.remove_callback = remove_callback
        self.auction_callback = auction_callback
        self.place_bid_callback = place_bid_callback
        self.end_auction_callback = end_auction_callback
        self.is_auction = is_auction
        self.image_label = None
        self.create_widgets()
        self.bind("<Button-1>", self.on_card_click)
        for child in self.winfo_children():
            child.bind("<Button-1>", self.on_card_click)

    def create_widgets(self):
        if self.is_auction:
            # For auction NFTs, nft_data has auction info and NFT details
            (self.auction_id, self.nft_id, self.seller_id, self.min_bid_price, self.current_bid, 
             self.current_bidder, self.name, self.description, self.price, self.image_path, self.seller_email) = self.nft_data
        else:
            if len(self.nft_data) == 6:
                self.nft_id, self.user_id, self.name, self.description, self.price, self.image_path = self.nft_data
            else:
                self.nft_id, self.name, self.description, self.price, self.image_path = self.nft_data
                self.user_id = None

        img_frame = tk.Frame(self)
        img_frame.pack(side="left", padx=(0,10))
        self.image_label = tk.Label(img_frame, cursor="hand2")
        self.load_image(self.image_path)
        self.image_label.pack()
        self.image_label.bind("<Button-1>", self.show_details)

        text_frame = tk.Frame(self)
        text_frame.pack(side="left", fill="both", expand=True)

        self.name_label = tk.Label(text_frame, text=self.name, font=("Arial", 14, "bold"), cursor="hand2", fg="blue")
        self.name_label.pack(anchor="w")
        self.name_label.bind("<Button-1>", self.show_details)

        desc_label = tk.Label(text_frame, text=self.description if self.description else "No description", wraplength=350, justify="left")
        desc_label.pack(anchor="w", pady=(5,10))

        if self.is_auction:
            current_bid_text = f"Current Bid: ${self.current_bid:.2f}" if self.current_bid else f"Minimum Bid: ${self.min_bid_price:.2f}"
            price_label = tk.Label(text_frame, text=current_bid_text, font=("Arial", 12, "bold"), fg='red')
        else:
            price_label = tk.Label(text_frame, text=f"Price: ${self.price:.2f}", font=("Arial", 12, "bold"))
        price_label.pack(anchor="e")

        btn_frame = tk.Frame(text_frame)
        btn_frame.pack(anchor="e", pady=(5,0))

        if self.is_auction:
            if self.place_bid_callback:
                bid_btn = tk.Button(btn_frame, text="Place Bid", command=lambda: self.place_bid_callback(self.auction_id, self.min_bid_price, self.current_bid))
                bid_btn.pack(side="right", padx=5)
            if self.end_auction_callback:
                end_btn = tk.Button(btn_frame, text="End Auction", command=lambda: self.end_auction_callback(self.auction_id))
                end_btn.pack(side="right", padx=5)
        else:
            if self.buy_callback:
                buy_btn = tk.Button(btn_frame, text="Buy", command=lambda: self.buy_callback(self.nft_id, self.price))
                buy_btn.pack(side="right", padx=5)
            if self.sell_callback:
                sell_btn = tk.Button(btn_frame, text="Sell", command=lambda: self.sell_callback(self.nft_id, self.price))
                sell_btn.pack(side="right", padx=5)
            if self.auction_callback:
                auction_btn = tk.Button(btn_frame, text="Auction", command=lambda: self.auction_callback(self.nft_id))
                auction_btn.pack(side="right", padx=5)
            if self.remove_callback:
                remove_btn = tk.Button(btn_frame, text="Remove", command=lambda: self.remove_callback(self.nft_id))
                remove_btn.pack(side="right", padx=5)

    def load_image(self, image_path):
        default_size = (100, 100)
        try:
            if image_path and os.path.exists(image_path):
                img = Image.open(image_path)
                img = ImageOps.exif_transpose(img)
                img.thumbnail(default_size)
                self.photo = ImageTk.PhotoImage(img)
            else:
                img = Image.new('RGB', default_size, color='gray')
                self.photo = ImageTk.PhotoImage(img)
        except Exception:
            img = Image.new('RGB', default_size, color='gray')
            self.photo = ImageTk.PhotoImage(img)
        self.image_label.configure(image=self.photo)
        self.image_label.image = self.photo

    def show_details(self, event=None):
        detail_win = tk.Toplevel(self)
        detail_win.title(self.name)
        detail_win.geometry("400x500")
        detail_win.resizable(False, False)

        # Load larger image if possible
        img_label = tk.Label(detail_win)
        img_label.pack(pady=10)
        default_size = (350, 350)
        try:
            if self.image_path and os.path.exists(self.image_path):
                img = Image.open(self.image_path)
                img = ImageOps.exif_transpose(img)
                img.thumbnail(default_size)
                photo = ImageTk.PhotoImage(img)
            else:
                img = Image.new('RGB', default_size, color='gray')
                photo = ImageTk.PhotoImage(img)
        except Exception:
            img = Image.new('RGB', default_size, color='gray')
            photo = ImageTk.PhotoImage(img)
        img_label.configure(image=photo)
        img_label.image = photo

        tk.Label(detail_win, text=self.name, font=("Arial", 16, "bold")).pack(pady=5)
        tk.Label(detail_win, text=self.description if self.description else "No description provided.", wraplength=380, justify="left").pack(pady=5)
        if self.is_auction:
            current_bid_text = f"Current Bid: ${self.current_bid:.2f}" if self.current_bid else f"Minimum Bid: ${self.min_bid_price:.2f}"
            tk.Label(detail_win, text=current_bid_text, font=("Arial", 14, "bold"), fg='red').pack(pady=5)
        else:
            tk.Label(detail_win, text=f"Price: ${self.price:.2f}", font=("Arial", 14, "bold")).pack(pady=5)

    def on_card_click(self, event):
        pass

class NFTMarketplaceApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("NFT Marketplace")
        self.geometry("900x650")
        self.resizable(False, False)
        self.db = DatabaseManager(DB_FILE)
        self.current_user_id = None

        self.signup_frame = SignupFrame(self, self.show_login_frame, self.db)
        self.signup_frame.pack(fill="both", expand=True)
        self.login_frame = LoginFrame(self, self.show_signup_frame, self.db)
        self.nft_frame = NFTMainFrame(self, self.db)

    def show_login_frame(self):
        self.signup_frame.pack_forget()
        self.nft_frame.pack_forget()
        self.login_frame.pack(fill="both", expand=True)

    def show_signup_frame(self):
        self.login_frame.pack_forget()
        self.nft_frame.pack_forget()
        self.signup_frame.pack(fill="both", expand=True)

    def show_nft_frame(self, user_id):
        self.current_user_id = user_id
        self.login_frame.pack_forget()
        self.signup_frame.pack_forget()
        self.nft_frame.load_user(user_id)
        self.nft_frame.pack(fill="both", expand=True)

    def on_closing(self):
        self.db.close()
        self.destroy()

class SignupFrame(tk.Frame):
    def __init__(self, parent, show_login_callback, db):
        super().__init__(parent)
        self.show_login_callback = show_login_callback
        self.db = db

        tk.Label(self, text="Sign Up", font=("Arial", 18, "bold")).pack(pady=15)
        tk.Label(self, text="Email:").pack(anchor="w", padx=60)
        self.email_entry = tk.Entry(self, width=40)
        self.email_entry.pack(padx=60, pady=5)
        tk.Label(self, text="Password:").pack(anchor="w", padx=60)
        self.password_entry = tk.Entry(self, width=40, show="*")
        self.password_entry.pack(padx=60, pady=5)
        tk.Label(self, text="Confirm Password:").pack(anchor="w", padx=60)
        self.confirm_password_entry = tk.Entry(self, width=40, show="*")
        self.confirm_password_entry.pack(padx=60, pady=5)

        tk.Button(self, text="Sign Up", width=20, command=self.signup).pack(pady=15)
        tk.Button(self, text="Back to Login", width=20, command=self.show_login_callback).pack()

    def signup(self):
        email = self.email_entry.get().strip()
        password = self.password_entry.get()
        confirm_password = self.confirm_password_entry.get()

        if not email:
            messagebox.showerror("Error", "Email is required.")
            return
        if not password:
            messagebox.showerror("Error", "Password is required.")
            return
        if password != confirm_password:
            messagebox.showerror("Error", "Passwords do not match.")
            return

        hashed_pw = self.db.hash_password(password)
        success = self.db.add_user(email, hashed_pw)
        if success:
            messagebox.showinfo("Success", f"Account created successfully for {email}. Please login.")
            self.email_entry.delete(0, tk.END)
            self.password_entry.delete(0, tk.END)
            self.confirm_password_entry.delete(0, tk.END)
            self.show_login_callback()
        else:
            messagebox.showerror("Error", "This email is already registered. Please login or use another email.")

class LoginFrame(tk.Frame):
    def __init__(self, parent, show_signup_callback, db):
        super().__init__(parent)
        self.show_signup_callback = show_signup_callback
        self.db = db

        tk.Label(self, text="Login", font=("Arial", 18, "bold")).pack(pady=15)
        tk.Label(self, text="Email:").pack(anchor="w", padx=60)
        self.email_entry = tk.Entry(self, width=40)
        self.email_entry.pack(padx=60, pady=5)
        tk.Label(self, text="Password:").pack(anchor="w", padx=60)
        self.password_entry = tk.Entry(self, width=40, show="*")
        self.password_entry.pack(padx=60, pady=5)

        tk.Button(self, text="Login", width=20, command=self.login).pack(pady=15)
        tk.Button(self, text="Back to Signup", width=20, command=self.show_signup_callback).pack()

    def login(self):
        email = self.email_entry.get().strip()
        password = self.password_entry.get()

        if not email:
            messagebox.showerror("Error", "Email is required.")
            return
        if not password:
            messagebox.showerror("Error", "Password is required.")
            return

        is_valid, user_id = self.db.verify_user(email, password)
        if is_valid:
            messagebox.showinfo("Success", f"Welcome {email}!")
            self.email_entry.delete(0, tk.END)
            self.password_entry.delete(0, tk.END)
            self.master.show_nft_frame(user_id)
        else:
            messagebox.showerror("Error", "Invalid email or password.")

class NFTMainFrame(tk.Frame):
    def __init__(self, parent, db):
        super().__init__(parent)
        self.db = db
        self.user_id = None

        # Header
        header_frame = tk.Frame(self)
        header_frame.pack(fill="x", pady=10, padx=10)

        tk.Label(header_frame, text="NFT Marketplace", font=("Arial", 20, "bold")).pack(side="left")

        self.wallet_balance_var = tk.StringVar(value="Wallet Balance: $0.00")
        self.wallet_balance_label = tk.Label(header_frame, textvariable=self.wallet_balance_var, font=("Arial", 14, "bold"))
        self.wallet_balance_label.pack(side="left", padx=20)

        tk.Button(header_frame, text="View Wallet & Transactions", command=self.show_wallet).pack(side="right", padx=5)
        tk.Button(header_frame, text="Logout", command=self.logout).pack(side="right")

        # Search bar for Buy NFTs tab
        search_frame = tk.Frame(self)
        search_frame.pack(fill="x", pady=5, padx=15)

        tk.Label(search_frame, text="Search NFTs:", font=("Arial", 11)).pack(side="left")
        self.search_var = tk.StringVar()
        self.search_entry = tk.Entry(search_frame, textvariable=self.search_var, width=30)
        self.search_entry.pack(side="left", padx=8)
        self.search_entry.bind("<KeyRelease>", self.on_search)

        # Notebook
        self.notebook = ttk.Notebook(self)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=10)

        self.buying_tab = tk.Frame(self.notebook)
        self.selling_tab = tk.Frame(self.notebook)
        self.auction_tab = tk.Frame(self.notebook)  # New auction tab

        self.notebook.add(self.buying_tab, text="Buy NFTs")
        self.notebook.add(self.selling_tab, text="Sell NFTs")
        self.notebook.add(self.auction_tab, text="Auctions")

        # Add NFT buttons
        tk.Button(self.buying_tab, text="Add NFT", command=self.add_nft).pack(pady=5)
        tk.Button(self.selling_tab, text="Add NFT", command=self.add_nft).pack(pady=5)
        tk.Button(self.auction_tab, text="Add NFT", command=self.add_nft).pack(pady=5)

        self.buying_canvas, self.buying_scrollable_frame = self.setup_scrollable_frame(self.buying_tab)
        self.selling_canvas, self.selling_scrollable_frame = self.setup_scrollable_frame(self.selling_tab)
        self.auction_canvas, self.auction_scrollable_frame = self.setup_scrollable_frame(self.auction_tab)

    def setup_scrollable_frame(self, parent):
        canvas = tk.Canvas(parent, height=460)
        scrollbar = ttk.Scrollbar(parent, orient="vertical", command=canvas.yview)
        scrollable_frame = ttk.Frame(canvas)

        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )

        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        # Enable mousewheel scrolling on Windows and Mac
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1*(event.delta/120)), "units")
        canvas.bind_all("<MouseWheel>", _on_mousewheel)  # Windows
        canvas.bind_all("<Button-4>", lambda e: canvas.yview_scroll(-1, "units"))  # Linux scroll up
        canvas.bind_all("<Button-5>", lambda e: canvas.yview_scroll(1, "units"))   # Linux scroll down

        return canvas, scrollable_frame

    def on_search(self, event=None):
        self.refresh_marketplace()

    def load_user(self, user_id):
        self.user_id = user_id
        self.update_wallet_balance()
        self.refresh_marketplace()
        self.refresh_user_nfts()
        self.refresh_auctions()
        self.notebook.select(0)  # focus on Buy NFTs tab

    def update_wallet_balance(self):
        wallet = self.db.get_wallet(self.user_id)
        if wallet:
            self.wallet_balance_var.set(f"Wallet Balance: ${wallet['balance']:.2f}")
        else:
            self.wallet_balance_var.set("Wallet Balance: $0.00")

    def refresh_marketplace(self):
        # Clear existing
        for widget in self.buying_scrollable_frame.winfo_children():
            widget.destroy()

        filter_text = self.search_var.get().strip()
        nfts = self.db.get_nfts(filter_text=filter_text)
        if not nfts:
            tk.Label(self.buying_scrollable_frame, text="No NFTs found.", font=("Arial", 12)).pack(pady=10)
            return

        for nft in nfts:
            NFTCard(
                self.buying_scrollable_frame,
                nft,
                buy_callback=self.buy_nft,
                sell_callback=self.sell_nft_with_prompt,
                remove_callback=(self.remove_nft if nft["user_id"] == self.user_id else None),
                auction_callback=(self.auction_nft if nft["user_id"] == self.user_id else None),
            ).pack(fill="x", padx=10, pady=5, expand=True)

    def refresh_user_nfts(self):
        for widget in self.selling_scrollable_frame.winfo_children():
            widget.destroy()

        nfts = self.db.get_user_nfts(self.user_id)
        if not nfts:
            tk.Label(self.selling_scrollable_frame, text="No NFTs to sell.", font=("Arial", 12)).pack(pady=10)
            return

        for nft in nfts:
            NFTCard(
                self.selling_scrollable_frame,
                nft,
                buy_callback=self.buy_nft,
                sell_callback=self.sell_nft_with_prompt,
                remove_callback=self.remove_nft,
                auction_callback=self.auction_nft,
            ).pack(fill="x", padx=10, pady=5, expand=True)

    def refresh_auctions(self):
        for widget in self.auction_scrollable_frame.winfo_children():
            widget.destroy()

        auctions = self.db.get_active_auctions()
        if not auctions:
            tk.Label(self.auction_scrollable_frame, text="No active auctions.", font=("Arial", 12)).pack(pady=10)
            return

        for auction in auctions:
            NFTCard(
                self.auction_scrollable_frame,
                auction,
                place_bid_callback=self.place_bid,
                end_auction_callback=self.end_auction,
                is_auction=True
            ).pack(fill="x", padx=10, pady=5, expand=True)

    def buy_nft(self, nft_id, price):
        wallet = self.db.get_wallet(self.user_id)
        if not wallet:
            messagebox.showerror("Wallet Error", "Failed to retrieve wallet info.")
            return

        # Check ownership: can't buy own NFT
        cursor = self.db.conn.cursor()
        cursor.execute("SELECT user_id FROM nfts WHERE id = ?", (nft_id,))
        owner_row = cursor.fetchone()
        cursor.close()
        if owner_row and owner_row["user_id"] == self.user_id:
            messagebox.showwarning("Invalid Operation", "You cannot buy your own NFT.")
            return

        if price > wallet["balance"]:
            messagebox.showerror("Insufficient Balance", "You do not have enough balance to buy this NFT.")
            return

        confirm = messagebox.askyesno("Confirm Purchase", f"Do you want to buy this NFT for ${price:.2f}?")
        if not confirm:
            return

        new_buyer_balance = wallet["balance"] - price
        new_buyer_expenses = wallet["expenses"] + price

        # Get seller info and update seller wallet
        cursor = self.db.conn.cursor()
        cursor.execute("SELECT user_id FROM nfts WHERE id = ?", (nft_id,))
        seller_info = cursor.fetchone()
        seller_id = None
        if seller_info:
            seller_id = seller_info["user_id"]
        cursor.execute("SELECT balance, total_expenses, total_profit_loss FROM wallets WHERE user_id = ?", (seller_id,))
        seller_wallet = cursor.fetchone()
        cursor.close()

        if seller_wallet is None:
            messagebox.showerror("Wallet Error", "Failed to retrieve seller wallet info.")
            return

        new_seller_balance = seller_wallet["balance"] + price
        new_seller_profit_loss = seller_wallet["total_profit_loss"] + price

        # Update wallets
        success_update_buyer = self.db.update_wallet(self.user_id, balance=new_buyer_balance, expenses=new_buyer_expenses)
        success_update_seller = self.db.update_wallet(seller_id, balance=new_seller_balance, profit_loss=new_seller_profit_loss)

        if success_update_buyer and success_update_seller:
            # Transfer ownership: remove old nft and re-add it with current user as owner
            cursor = self.db.conn.cursor()
            cursor.execute("SELECT name, description, image_path FROM nfts WHERE id = ?", (nft_id,))
            nft_details = cursor.fetchone()
            cursor.close()

            # Remove old NFT
            if not self.db.remove_nft(nft_id):
                messagebox.showerror("Error", "Purchase failed during transfer.")
                return

            # Add NFT to new owner
            if not self.db.add_nft(self.user_id, nft_details["name"], nft_details["description"], price, nft_details["image_path"]):
                messagebox.showerror("Error", "Purchase failed during transfer.")
                return

            # Log transactions for buyer and seller
            self.db.add_transaction(self.user_id, None, "buy", price)
            self.db.add_transaction(seller_id, nft_id, "sell", price)

            messagebox.showinfo("Purchase Successful", f"You bought the NFT for ${price:.2f}.")
            self.update_wallet_balance()
            self.refresh_marketplace()
            self.refresh_user_nfts()
            self.refresh_auctions()
        else:
            messagebox.showerror("Error", "Purchase failed.")

    def sell_nft_with_prompt(self, nft_id, current_price):
        price = simpledialog.askfloat("Set Sell Price", f"Current NFT price is ${current_price:.2f}.\nEnter your sell price:")
        if price is None or price < 0:
            messagebox.showerror("Input Error", "Sell price must be a non-negative number.")
            return
        self.sell_nft(nft_id, price)

    def sell_nft(self, nft_id, price):
        wallet = self.db.get_wallet(self.user_id)
        if not wallet:
            messagebox.showerror("Wallet Error", "Failed to retrieve wallet info.")
            return

        cursor = self.db.conn.cursor()
        cursor.execute("SELECT user_id FROM nfts WHERE id = ?", (nft_id,))
        owner_row = cursor.fetchone()
        cursor.close()
        if not (owner_row and owner_row["user_id"] == self.user_id):
            messagebox.showwarning("Invalid Operation", "You can only sell your own NFTs.")
            return

        confirm = messagebox.askyesno("Confirm Sale", f"Do you want to sell this NFT for ${price:.2f}? This will remove NFT from your inventory.")
        if not confirm:
            return

        new_seller_balance = wallet["balance"] + price
        new_seller_profit_loss = wallet["profit_loss"] + price

        success_update = self.db.update_wallet(self.user_id, balance=new_seller_balance, profit_loss=new_seller_profit_loss)
        success_remove = self.db.remove_nft(nft_id)

        if success_update and success_remove:
            self.db.add_transaction(self.user_id, nft_id, "sell", price)
            messagebox.showinfo("Sale Successful", f"You sold the NFT for ${price:.2f}.")
            self.update_wallet_balance()
            self.refresh_marketplace()
            self.refresh_user_nfts()
            self.refresh_auctions()
        else:
            messagebox.showerror("Error", "Sale failed.")

    def auction_nft(self, nft_id):
        min_bid = simpledialog.askfloat("Start Auction", "Enter minimum starting bid price:")
        if min_bid is None or min_bid < 0:
            messagebox.showerror("Input Error", "Minimum bid price must be a non-negative number.")
            return

        existing_auction = self.db.get_auction_by_nft(nft_id)
        if existing_auction:
            messagebox.showerror("Auction Error", "This NFT is already on auction.")
            return

        confirm = messagebox.askyesno("Confirm Auction", f"Start auction for this NFT with minimum bid price ${min_bid:.2f}?")
        if not confirm:
            return

        success = self.db.add_auction(nft_id, self.user_id, min_bid)
        if success:
            messagebox.showinfo("Auction Started", "NFT is now on auction.")
            self.refresh_marketplace()
            self.refresh_user_nfts()
            self.refresh_auctions()
        else:
            messagebox.showerror("Error", "Failed to start auction. It may already exist.")

    def place_bid(self, auction_id, min_bid, current_bid):
        wallet = self.db.get_wallet(self.user_id)
        if not wallet:
            messagebox.showerror("Wallet Error", "Failed to retrieve wallet info.")
            return
        max_bid = wallet["balance"]
        prompt_msg = f"Enter your bid amount. Minimum bid is ${min_bid:.2f}."
        if current_bid is not None:
            prompt_msg += f" Current highest bid is ${current_bid:.2f}."
        prompt_msg += f"\nYour available balance: ${max_bid:.2f}."
        bid_amount = simpledialog.askfloat("Place Bid", prompt_msg)
        if bid_amount is None or bid_amount < 0:
            messagebox.showerror("Input Error", "Bid amount must be a non-negative number.")
            return
        if bid_amount > max_bid:
            messagebox.showerror("Insufficient Balance", "You do not have enough balance for this bid.")
            return

        success, msg = self.db.add_bid(auction_id, self.user_id, bid_amount)
        if success:
            messagebox.showinfo("Bid Placed", msg)
            self.refresh_auctions()
        else:
            messagebox.showerror("Bid Failed", msg)

    def end_auction(self, auction_id):
        confirm = messagebox.askyesno("End Auction", "Are you sure you want to end this auction?")
        if not confirm:
            return
        success, msg = self.db.end_auction(auction_id)
        if success:
            messagebox.showinfo("Auction Ended", msg)
            self.update_wallet_balance()
            self.refresh_marketplace()
            self.refresh_user_nfts()
            self.refresh_auctions()
        else:
            messagebox.showerror("Error", msg)

    def remove_nft(self, nft_id):
        confirm = messagebox.askyesno("Confirm Remove", "Are you sure you want to remove this NFT permanently?")
        if confirm:
            success = self.db.remove_nft(nft_id)
            if success:
                messagebox.showinfo("Removed", "NFT removed successfully.")
                self.refresh_marketplace()
                self.refresh_user_nfts()
                self.refresh_auctions()
            else:
                messagebox.showerror("Error", "Failed to remove NFT.")

    def add_nft(self):
        name = simpledialog.askstring("NFT Name", "Enter NFT Name:")
        if not name:
            messagebox.showerror("Input Error", "NFT Name is required.")
            return
        description = simpledialog.askstring("NFT Description", "Enter NFT Description:")

        try:
            price = simpledialog.askfloat("NFT Price", "Enter NFT Price:")
            if price is None or price < 0:
                messagebox.showerror("Input Error", "Price must be a non-negative number.")
                return
        except Exception:
            messagebox.showerror("Input Error", "Invalid price entered.")
            return

        img_path = filedialog.askopenfilename(title="Select NFT Image",
                                              filetypes=[("Image Files", "*.png *.jpg *.jpeg *.gif *.bmp")])
        saved_img_path = None
        if img_path:
            try:
                ext = os.path.splitext(img_path)[1]
                base_name = f"nft_{self.user_id}_{int(hashlib.sha256((img_path + str(name)).encode()).hexdigest(), 16) % (10 ** 8)}{ext}"
                saved_img_path = os.path.join(NFT_IMAGE_DIR, base_name)
                shutil.copyfile(img_path, saved_img_path)
            except Exception as e:
                messagebox.showerror("Image Error", f"Failed to save image: {e}")
                saved_img_path = None

        if self.user_id is None:
            messagebox.showerror("Error", "User not logged in.")
            return

        success = self.db.add_nft(self.user_id, name, description, price, saved_img_path)
        if success:
            messagebox.showinfo("Success", "NFT added successfully!")
            self.refresh_user_nfts()
            self.refresh_marketplace()
            self.refresh_auctions()
        else:
            messagebox.showerror("Error", "Failed to add NFT.")

    def show_wallet(self):
        if self.user_id is None:
            messagebox.showerror("Error", "User not logged in.")
            return
        wallet = self.db.get_wallet(self.user_id)
        if not wallet:
            messagebox.showerror("Error", "Failed to fetch wallet data.")
            return

        transactions = self.db.get_transactions(self.user_id)

        wallet_window = tk.Toplevel(self)
        wallet_window.title("Wallet & Transactions")
        wallet_window.geometry("400x500")
        wallet_window.resizable(False, False)

        tk.Label(wallet_window, text="Wallet Information", font=("Arial", 16, "bold")).pack(pady=10)
        tk.Label(wallet_window, text=f"Available Balance: ${wallet['balance']:.2f}", font=("Arial", 14)).pack(pady=5)
        tk.Label(wallet_window, text=f"Total Expenses: ${wallet['expenses']:.2f}", font=("Arial", 14)).pack(pady=5)
        tk.Label(wallet_window, text=f"Profit / Loss: ${wallet['profit_loss']:.2f}", font=("Arial", 14)).pack(pady=5)

        sep = ttk.Separator(wallet_window, orient='horizontal')
        sep.pack(fill='x', pady=10)

        tk.Label(wallet_window, text="Transaction History", font=("Arial", 14, "bold")).pack()

        trans_frame = tk.Frame(wallet_window)
        trans_frame.pack(fill="both", expand=True, padx=10, pady=5)

        columns = ("type", "amount", "timestamp")
        tree = ttk.Treeview(trans_frame, columns=columns, show="headings")
        tree.heading("type", text="Type")
        tree.heading("amount", text="Amount")
        tree.heading("timestamp", text="Timestamp")
        tree.column("type", width=80)
        tree.column("amount", width=80)
        tree.column("timestamp", width=180)
        tree.pack(side="left", fill="both", expand=True)

        scroll = ttk.Scrollbar(trans_frame, orient="vertical", command=tree.yview)
        tree.configure(yscrollcommand=scroll.set)
        scroll.pack(side="right", fill="y")

        for t in transactions:
            type_, amount, timestamp, nft_id = t["type"], t["amount"], t["timestamp"], t["nft_id"]
            ts = timestamp if isinstance(timestamp, str) else timestamp.strftime("%Y-%m-%d %H:%M:%S")
            tree.insert("", "end", values=(type_.capitalize(), f"${amount:.2f}", ts))

        tk.Button(wallet_window, text="Close", command=wallet_window.destroy).pack(pady=10)

    def logout(self):
        confirm = messagebox.askyesno("Logout", "Are you sure you want to logout?")
        if confirm:
            self.pack_forget()
            self.master.current_user_id = None
            self.pack_forget()
            self.master.signup_frame.pack(fill="both", expand=True)

if __name__ == "__main__":
    app = NFTMarketplaceApp()
    app.protocol("WM_DELETE_WINDOW", app.on_closing)
    app.mainloop()


/tmp/ipykernel_10189/2394377124.py:267: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO transactions (user_id, nft_id, type, amount, timestamp) VALUES (?, ?, ?, ?, ?)",
